# Optimize polynomial prediction

In [ ]:
%matplotlib ipympl

import scipy.linalg as sci_lin
from helper import *

In [ ]:
def propogate_pos(x0, E, n):
    def scan_body(x0, _):
        x1 = E @ x0
        return x1, x1[0]

    _, res = jax.lax.scan(scan_body, x0, length=n)
    return res.flatten()

def diff(x, dt):
    return jnp.diff(x) / dt

def taylor_coeffs(hist, dt):
    assert hist.shape == (4,)
    a0 = hist[3]
    d = functools.partial(diff, dt=dt)
    a1 = d(hist[2:])
    a2 = d(d(hist[1:])) / 2
    a3 = d(d(d(hist))) / 6
    return a0, a1, a2, a3

def taylor_eval(hist, x, dt):
    a0, a1, a2, a3 = taylor_coeffs(hist, dt)
    return a0 + a1 * x + a2 * x**2 + a3 * x**3

def taylor_evalp(hist, x, dt):
    a0, a1, a2, a3 = taylor_coeffs(hist, dt)
    f0 = a0 + a1 * x + a2 * x**2 + a3 * x**3
    f1 = a1 + 2 * a2 * x + 3 * a3 * x**2
    f2 = 2 * a2 + 6 * a3 * x
    return jnp.array([f0, f1, f2])

# @functools.partial(jax.jit, static_argnames=["n_taylor", "n"])
def pred_hist(n_taylor, n, dt, E, hist):
    assert n_taylor < n
    t = jnp.arange(1, n_taylor + 1, dtype=float) * dt
    res0 = taylor_eval(hist, t, dt)
    x0 = taylor_evalp(hist, t[-1], dt)
    res1 = propogate_pos(x0, E, n - n_taylor)
    res = jnp.concatenate([res0, res1])
    return res

def make_E(alpha):
    alpha = np.eye(3) * -alpha
    A = np.array([[0, 1, 0], [0, 0, 1], [0, 0, 0]], dtype=float)
    B = np.array([[0], [0], [1]], dtype=float)
    Q = np.diag([1e0, 1e0, 1e0])
    R = np.array([[1e-0]], dtype=float)
    K, _, _ = ct.lqr(A - alpha, B, Q, R)
    E = sci_lin.expm((A - B @ K) * spec.dt)
    return E

@functools.partial(jax.jit, static_argnames=["n_taylor"])
def cost_fun(y_data, E, n_taylor):
    mixed_pred = functools.partial(pred_hist, n_taylor, spec.n, spec.dt, E)
    err = pred_err(lambda hist, _, __: mixed_pred(hist), y_data, 4)
    return err

In [ ]:
lti_int, x_data, y_data, ctrl_data = get_data(0)

In [ ]:
res = {}
alphas = np.arange(0.2, 5.0, step=0.05)
n_taylors = range(10, 75, 2)

for j in tqdm.tqdm(range(len(n_taylors))):
    for i in range(len(alphas)):
        err = cost_fun(y_data, make_E(alphas[i]), n_taylors[j])
        res[(i, j)] = err

In [ ]:
s_res = sorted([(elem, key) for key, elem in list(res.items())])
float(np.round(alphas[s_res[0][1][0]], decimals=2)), n_taylors[s_res[0][1][1]], float(s_res[0][0])